# Week 06: Stats for Business: Is the Price Right?

You just joined the pricing team at a car-listing marketplace. Your manager has a few questions and one rule: **every claim needs a number, and every number needs a reason to trust it.**

**Data:** `cars.csv`: about 12,000 car trims (1990–2017) with specs and sticker price (MSRP). Source: [Car Features and MSRP (Kaggle)](https://www.kaggle.com/datasets/CooperUnion/cardataset). It's real data, so it's messy.

**What you'll practice** (all from this week's lecture):
- Cleaning before describing
- Mean vs median, skew, log transforms
- z-scores
- Bootstrap confidence intervals
- One- and two-sample tests, and picking the right one
- Confounding, and significant vs important

**How to answer:** Fill in the code cells, then write 1–3 sentences in each **Answer:** cell. Code without an interpretation is only half the job.

Plan for about 2–3 hours. 🚀 questions are optional stretch.

## Setup

In [3]:
# run this first if you dont have them installed in your env
!pip install pandas numpy matplotlib scipy

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", None)
rng = np.random.default_rng(42)  # use rng for all random sampling so your results are reproducible

df = pd.read_csv("/workspaces/ds-fall-2026/Week-05-Business-Stats-Analytics/data/cars.csv")
print(df.shape)
df.head()

(11914, 15)


,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Compact,Convertible,28,18,3916,34500


---
## Part 1: Can you trust this data?

### Q1. Find the problems before you average anything

This dataset has **at least five** real problems. Find them, fix or flag each one, and build a cleaned DataFrame called `clean`. You'll use `clean` for the rest of the notebook.

Use your Week 02 cleaning checklist. Some nudges, in no particular order:
- Count things twice.
- The most common value in a column can be more suspicious than the largest one.
- Would you print every max value in a sales brochure?
- Missing values usually have a reason. *Who* is missing?
- Is every number in a column measured in the same unit?
- Does every column vary at the level you'd expect?
- Not everything weird is wrong.

For each problem, decide: **drop it, set it to NaN, flag it, or leave it**. Dropping whole rows is the last resort, because a row with a bad price may still have a good MPG.

In [63]:
# Explore here. Add as many cells as you need.
# Useful: .duplicated(), .value_counts(), .describe(), .isna().sum(), .nlargest(), .groupby(...).nunique()
print(df.head())
print(df.info())


  Make       Model  Year             Engine Fuel Type  Engine HP  \
0  BMW  1 Series M  2011  premium unleaded (required)      335.0   
1  BMW    1 Series  2011  premium unleaded (required)      300.0   
2  BMW    1 Series  2011  premium unleaded (required)      300.0   
3  BMW    1 Series  2011  premium unleaded (required)      230.0   
4  BMW    1 Series  2011  premium unleaded (required)      230.0   

   Engine Cylinders Transmission Type     Driven_Wheels  Number of Doors  \
0               6.0            MANUAL  rear wheel drive              2.0   
1               6.0            MANUAL  rear wheel drive              2.0   
2               6.0            MANUAL  rear wheel drive              2.0   
3               6.0            MANUAL  rear wheel drive              2.0   
4               6.0            MANUAL  rear wheel drive              2.0   

  Vehicle Size Vehicle Style  highway MPG  city mpg  Popularity   MSRP  
0      Compact         Coupe           26        19        39

In [ ]:
# inspecting duplicated rows.
copy = df.copy()
copy["duplicated_results"] = df.duplicated(keep=False)
copy["group_ID"] = (copy.groupby(list(df.columns), dropna=False).ngroup())
copy.sort_values(["duplicated_results", "group_ID"], ascending=False)

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP,duplicated_results,group_ID
11601,Volvo,XC60,2017,regular unleaded,302.0,4.0,AUTOMATIC,all wheel drive,4.0,Midsize,4dr SUV,29,20,870,46350,True,11157
11603,Volvo,XC60,2017,regular unleaded,302.0,4.0,AUTOMATIC,all wheel drive,4.0,Midsize,4dr SUV,29,20,870,46350,True,11157
11600,Volvo,XC60,2017,regular unleaded,240.0,4.0,AUTOMATIC,front wheel drive,4.0,Midsize,4dr SUV,30,23,870,40950,True,11156
11604,Volvo,XC60,2017,regular unleaded,240.0,4.0,AUTOMATIC,front wheel drive,4.0,Midsize,4dr SUV,30,23,870,40950,True,11156
8918,Volvo,S70,1998,regular unleaded,168.0,5.0,MANUAL,front wheel drive,4.0,Midsize,Sedan,27,17,870,2000,True,11038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2703,Acura,CL,2003,premium unleaded (required),225.0,6.0,AUTOMATIC,front wheel drive,2.0,Midsize,Coupe,27,17,204,28200,False,4
2699,Acura,CL,2002,premium unleaded (required),225.0,6.0,AUTOMATIC,front wheel drive,2.0,Midsize,Coupe,27,17,204,30030,False,3
2698,Acura,CL,2002,premium unleaded (required),225.0,6.0,AUTOMATIC,front wheel drive,2.0,Midsize,Coupe,27,17,204,28030,False,2
2696,Acura,CL,2001,premium unleaded (required),225.0,6.0,AUTOMATIC,front wheel drive,2.0,Midsize,Coupe,27,17,204,29980,False,1


In [61]:
df["MSRP"].describe()
sussyPriceCond = df["MSRP"] == 2000
df[sussyPriceCond].value_counts()

Make    Model      Year  Engine Fuel Type  Engine HP  Engine Cylinders  Transmission Type  Driven_Wheels      Number of Doors  Vehicle Size  Vehicle Style       highway MPG  city mpg  Popularity  MSRP
Dodge   RAM 150    1991  regular unleaded  125.0      6.0               UNKNOWN            rear wheel drive   2.0              Large         Regular Cab Pickup  17           12        1851        2000    8
GMC     S-15       1990  regular unleaded  105.0      4.0               MANUAL             rear wheel drive   2.0              Compact       Regular Cab Pickup  25           21        549         2000    7
                                           160.0      6.0               MANUAL             four wheel drive   2.0              Compact       Regular Cab Pickup  19           15        549         2000    6
Ford    Bronco II  1990  regular unleaded  140.0      6.0               MANUAL             rear wheel drive   2.0              Compact       2dr SUV             20           16     

In [62]:
df["Number of Doors"].value_counts()

Number of Doors
4.0    8353
2.0    3160
3.0     395
Name: count, dtype: int64

In [68]:
df.info()
nullValuesCond = df["Engine HP"].isna()
electricEngineCond = df["Engine Fuel Type"] == "electric"
df[nullValuesCond & electricEngineCond]
df[nullValuesCond & ~electricEngineCond]

<class 'pandas.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  str    
 1   Model              11914 non-null  str    
 2   Year               11914 non-null  int64  
 3   Engine Fuel Type   11911 non-null  str    
 4   Engine HP          11845 non-null  float64
 5   Engine Cylinders   11884 non-null  float64
 6   Transmission Type  11914 non-null  str    
 7   Driven_Wheels      11914 non-null  str    
 8   Number of Doors    11908 non-null  float64
 9   Vehicle Size       11914 non-null  str    
 10  Vehicle Style      11914 non-null  str    
 11  highway MPG        11914 non-null  int64  
 12  city mpg           11914 non-null  int64  
 13  Popularity         11914 non-null  int64  
 14  MSRP               11914 non-null  int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 1.4 MB


,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
2905,Lincoln,Continental,2017,premium unleaded (recommended),NaN,6.0,AUTOMATIC,all wheel drive,4.0,Large,Sedan,25,17,61,55915
2906,Lincoln,Continental,2017,premium unleaded (recommended),NaN,6.0,AUTOMATIC,front wheel drive,4.0,Large,Sedan,27,18,61,62915
2907,Lincoln,Continental,2017,premium unleaded (recommended),NaN,6.0,AUTOMATIC,front wheel drive,4.0,Large,Sedan,27,18,61,53915
2908,Lincoln,Continental,2017,premium unleaded (recommended),NaN,6.0,AUTOMATIC,all wheel drive,4.0,Large,Sedan,25,17,61,64915
4203,Ford,Escape,2017,regular unleaded,NaN,4.0,AUTOMATIC,front wheel drive,4.0,Compact,4dr SUV,30,23,5657,29100
4204,Ford,Escape,2017,regular unleaded,NaN,4.0,AUTOMATIC,all wheel drive,4.0,Compact,4dr SUV,28,22,5657,30850
4205,Ford,Escape,2017,regular unleaded,NaN,4.0,AUTOMATIC,all wheel drive,4.0,Compact,4dr SUV,28,22,5657,26850
4206,Ford,Escape,2017,regular unleaded,NaN,4.0,AUTOMATIC,front wheel drive,4.0,Compact,4dr SUV,30,23,5657,25100
4914,Ford,Freestar,2005,regular unleaded,NaN,6.0,AUTOMATIC,front wheel drive,4.0,Midsize,Passenger Minivan,22,16,5657,28030
4915,Ford,Freestar,2005,regular unleaded,NaN,6.0,AUTOMATIC,front wheel drive,4.0,Midsize,Passenger Minivan,22,16,5657,23930


In [74]:
df.info()
df[df["Engine Fuel Type"].isna()]
df[df["Engine Cylinders"].isna()]
df[df["Number of Doors"].isna()]


<class 'pandas.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  str    
 1   Model              11914 non-null  str    
 2   Year               11914 non-null  int64  
 3   Engine Fuel Type   11911 non-null  str    
 4   Engine HP          11845 non-null  float64
 5   Engine Cylinders   11884 non-null  float64
 6   Transmission Type  11914 non-null  str    
 7   Driven_Wheels      11914 non-null  str    
 8   Number of Doors    11908 non-null  float64
 9   Vehicle Size       11914 non-null  str    
 10  Vehicle Style      11914 non-null  str    
 11  highway MPG        11914 non-null  int64  
 12  city mpg           11914 non-null  int64  
 13  Popularity         11914 non-null  int64  
 14  MSRP               11914 non-null  int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 1.4 MB


,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
4666,Ferrari,FF,2013,premium unleaded (required),651.0,12.0,AUTOMATED_MANUAL,all wheel drive,NaN,Large,Coupe,16,11,2774,295000
6930,Tesla,Model S,2016,electric,NaN,0.0,DIRECT_DRIVE,all wheel drive,NaN,Large,Sedan,105,102,1391,79500
6931,Tesla,Model S,2016,electric,NaN,0.0,DIRECT_DRIVE,all wheel drive,NaN,Large,Sedan,101,98,1391,66000
6932,Tesla,Model S,2016,electric,NaN,0.0,DIRECT_DRIVE,all wheel drive,NaN,Large,Sedan,105,92,1391,134500
6933,Tesla,Model S,2016,electric,NaN,0.0,DIRECT_DRIVE,rear wheel drive,NaN,Large,Sedan,100,97,1391,74500
6934,Tesla,Model S,2016,electric,NaN,0.0,DIRECT_DRIVE,all wheel drive,NaN,Large,Sedan,107,101,1391,71000


In [82]:
df["highway MPG"].describe()
df["highway MPG"].value_counts()
df.sort_values("highway MPG", ascending=False)
audiAndA6Cond = (df["Make"] == "Audi") & (df["Model"] == "A6")
df[audiAndA6Cond]

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
1098,Audi,A6,2015,diesel,240.0,6.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,38,24,3105,58700
1099,Audi,A6,2015,premium unleaded (recommended),220.0,4.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,29,20,3105,50450
1100,Audi,A6,2015,premium unleaded (recommended),220.0,4.0,AUTOMATIC,front wheel drive,4.0,Midsize,Sedan,33,25,3105,44800
1101,Audi,A6,2015,premium unleaded (required),310.0,6.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,27,18,3105,60100
1102,Audi,A6,2015,premium unleaded (recommended),220.0,4.0,AUTOMATIC,front wheel drive,4.0,Midsize,Sedan,33,25,3105,48350
1103,Audi,A6,2015,premium unleaded (required),310.0,6.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,27,18,3105,56300
1104,Audi,A6,2015,premium unleaded (recommended),220.0,4.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,29,20,3105,46900
1105,Audi,A6,2015,diesel,240.0,6.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,38,24,3105,62500
1106,Audi,A6,2016,premium unleaded (recommended),252.0,4.0,AUTOMATIC,all wheel drive,4.0,Midsize,Sedan,32,22,3105,52100
1107,Audi,A6,2016,premium unleaded (recommended),252.0,4.0,AUTOMATED_MANUAL,front wheel drive,4.0,Midsize,Sedan,35,24,3105,49900


In [ ]:
dropped["Popularity"].describe()
dropped["Popularity"].value_counts()
dropped.groupby("Popularity")
# idk what I'm looking at. choosing something else.
display(dropped.sort_values("Popularity"))


,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP,sussyPrice,sussyEngines
1989,Spyker,C8,2009,premium unleaded (required),400.0,8.0,MANUAL,rear wheel drive,2.0,Compact,Coupe,18,13,2,209990,False,False
1990,Spyker,C8,2009,premium unleaded (required),400.0,8.0,MANUAL,rear wheel drive,2.0,Compact,Convertible,18,13,2,219990,False,False
4961,Genesis,G80,2017,regular unleaded,311.0,6.0,AUTOMATIC,rear wheel drive,4.0,Large,Sedan,28,18,21,41400,False,False
4963,Genesis,G80,2017,regular unleaded,311.0,6.0,AUTOMATIC,all wheel drive,4.0,Large,Sedan,25,16,21,43900,False,False
4962,Genesis,G80,2017,premium unleaded (recommended),420.0,8.0,AUTOMATIC,rear wheel drive,4.0,Large,Sedan,23,15,21,54550,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10062,Ford,Transit Wagon,2017,flex-fuel (unleaded/E85),275.0,6.0,AUTOMATIC,rear wheel drive,3.0,Midsize,Passenger Van,19,14,5657,34515,False,False
10063,Ford,Transit Wagon,2017,regular unleaded,275.0,6.0,AUTOMATIC,rear wheel drive,3.0,Midsize,Passenger Van,19,14,5657,38015,False,False
3970,Ford,Escape,2016,regular unleaded,168.0,4.0,AUTOMATIC,front wheel drive,4.0,Compact,4dr SUV,31,22,5657,23590,False,False
3971,Ford,Escape,2016,premium unleaded (recommended),178.0,4.0,AUTOMATIC,all wheel drive,4.0,Compact,4dr SUV,29,22,5657,31745,False,False


In [104]:
dropped["Transmission Type"].value_counts(dropna=False)
unknownTransmissionCond = (dropped["Transmission Type"] == "UNKNOWN")
dropped["unknownTransmission"] = unknownTransmissionCond
dropped.sort_values("unknownTransmission", ascending=False)
dropped.loc[unknownTransmissionCond, "Transmission Type"] = np.nan
print(dropped["unknownTransmission"].value_counts())



unknownTransmission
False    11194
Name: count, dtype: int64


In [100]:
# 1st solution.
dropped = df.copy()
dropped = dropped.drop_duplicates(keep='first').reset_index(drop=True)
dropped.info()
# 2nd solution.
dropped["sussyPrice"] = dropped["MSRP"] == 2000
# 3rd solution.
nullHPCond = dropped["Engine HP"].isna()
electricEngineCond = dropped["Engine Fuel Type"] == "electric"
dropped["sussyEngines"] = nullHPCond & ~electricEngineCond
#  4th solution.
sussyMPG = dropped["highway MPG"] == 354
dropped.loc[sussyMPG, "highway MPG"] = 34
# 5th solution.
dropped["unknownTransmission"] = unknownTransmissionCond
dropped.loc[unknownTransmissionCond, "Transmission Type"] = np.nan

<class 'pandas.DataFrame'>
RangeIndex: 11194 entries, 0 to 11193
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11194 non-null  str    
 1   Model              11194 non-null  str    
 2   Year               11194 non-null  int64  
 3   Engine Fuel Type   11191 non-null  str    
 4   Engine HP          11125 non-null  float64
 5   Engine Cylinders   11164 non-null  float64
 6   Transmission Type  11194 non-null  str    
 7   Driven_Wheels      11194 non-null  str    
 8   Number of Doors    11188 non-null  float64
 9   Vehicle Size       11194 non-null  str    
 10  Vehicle Style      11194 non-null  str    
 11  highway MPG        11194 non-null  int64  
 12  city mpg           11194 non-null  int64  
 13  Popularity         11194 non-null  int64  
 14  MSRP               11194 non-null  int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 1.3 MB


**Cleaning log:** fill in one row per problem.

| # | Problem | How I found it | Rows affected | What I did | Why |
|---|---|---|---|---|---|
| 1 | There were rows that contained the exact same information, column by column| df.duplicate(), I used some other commands to inspect the sussy rows. | 720 | df.copy().drop_duplicates() | Pretty sure I can safely rows that are identical to each other, since copies don't have any deeper meaning. However, if this database was about, idk, sales of cars, then theoretically a person could have bought a car of the same make, model, year, msrp, etc, more than once. In which case, I'd need to reevaluate whether I'd want to do the same in that scenario.|
| 2 | Some rows have an MSRP of $2,000 | df.describe() | 747 | created a new column, and flagged them as suspicious | idk if they're sussy or not. Idk how to distinguish whether $2,000 is a valid sticker price, or maybe it was in the past but isn't now.|
| 3 | Some rows are missing a listed horsepower value | Inspected df.info() and checked the non-null counts | 25 | flagged them as sussy. | I can buy that electric cars might not have a listed horsepower. However, non-electric cars, I think, should have it listed. But, idk anything abour cars so I'm just flagging it for now.|
| 4 | An audi was secretely an F1!  | df["highway MPG"].describe() | 1 | changed its value from 354 to 34 | I looked at similar makes and models, and another one has a highway mpg of 34. I figured it was a typo.|
| 5 | Cars were missing their transmission values | df["Transmission Type"].value_counts | 12 | replaced them with NAN | idk. I want to create null values, but idk if this is the correct way to do it. That is, if I want to meaningfully select from cars with different transmissions, then I want to exclude these cars without listed transmission values. Setting them to NAN seemed like a... choice.|

In [110]:
# Build your cleaned DataFrame. Start from a copy so df stays untouched.
clean = dropped.copy()
clean.columns = clean.columns.str.strip().str.lower().str.replace(" ", "_")  # freebie: snake_case names

In [109]:
# Checkpoint: run this before moving on.
print(clean.shape)
print(clean.isna().sum()[clean.isna().sum() > 0])
# Fewer than ~10,000 rows? You probably dropped too much. Go back and use NaN or a flag instead.

(11194, 18)
engine_fuel_type      3
engine_hp            69
engine_cylinders     30
transmission_type    12
number_of_doors       6
dtype: int64


---
## Part 2: Describe

### Q2. What does a "typical" car cost?

Your manager wants one number for "typical price" on the homepage.

a) Compute the mean and median MSRP. Which is bigger, and why?

b) Plot a histogram of MSRP, then a histogram of `np.log10(msrp)`. Describe each shape in a few words.

c) What percent of cars cost more than the mean?

d) Which number belongs on the homepage? Why?

In [ ]:
# a) mean and median

# b) two histograms side by side (plt.subplots(1, 2))

# c) share of cars above the mean

**Answer:**

### Q3. Fuel-efficient *for its size*?

Two 2017 cars:
- **Honda Civic** (Compact): 42 highway MPG
- **Volvo S90** (Large): 34 highway MPG

The Civic wins on raw MPG, but large cars are heavier. Which one is more impressive *for its class*?

a) For 2017 cars (only the MPG values you trust after Q1), compute the mean and standard deviation of highway MPG for each `vehicle_size`.

b) Compute each car's z-score within its own size class.

c) Which is more fuel-efficient for its class? How unusual is each one?

In [ ]:
# a) mean and std of highway_mpg by vehicle_size, 2017 only

# b) z = (value - class mean) / class std

**Answer:**

---
## Part 3: Uncertainty

### Q4. How sure are we about the median price?

The 2017 cars are one snapshot of the market. With a slightly different set of cars, the median would move. A bootstrap shows how much.

a) Compute the median MSRP of 2017 cars.

b) Bootstrap it: resample the 2017 prices **with replacement** 5,000 times and take the median each time. Plot the 5,000 medians and report the 95% interval (2.5th and 97.5th percentiles).

c) Repeat for 2017 cars from **one make** with fewer than 50 cars that year (your choice). Is the interval wider or narrower? Why?

d) Explain the interval from (b) in one sentence a manager would understand.

In [ ]:
def bootstrap_median(values, n_boot=5000):
    """Return an array of n_boot bootstrap medians."""
    # hint: rng.choice(values, size=len(values), replace=True)
    pass

# a, b)

# c)

**Answer:**

---
## Part 4: Testing

### Q5. Can marketing say "our 2017 compacts average over 30 MPG"?

a) Write H₀ and Hₐ. One-sided or two-sided? Why?

b) How many cars are in the sample, and what does the distribution look like? Is a t-test reasonable here?

c) Run a one-sample t-test at α = 0.05 (`stats.ttest_1samp(..., alternative="greater")`). State the conclusion in plain English.

d) **Independence check.** Tests assume each row is an independent observation. Count how many rows each `make` + `model` has in this sample. Then average to one row per model and rerun the test. What changed, and which version do you trust more?

In [ ]:
# b) sample: 2017 compact cars (MPG you trust); n, mean, histogram

# c) one-sample t-test vs 30

# d) rows per make+model, then one row per model and retest

**Answer:**

### Q6. Do manual cars cost less? (the big one)

A sales rep says: *"Manual cars are way cheaper. We should push manuals to budget shoppers."*

Compare `AUTOMATIC` vs `MANUAL` only.

a) Compare the median MSRP of the two groups.

b) Before believing it, compare the `year` of manual vs automatic cars. What do you notice?

c) Redo (a) using only 2015–2017 cars, then within each `vehicle_size`. For 2015–17 compacts, use the Mann–Whitney U test (`stats.mannwhitneyu(auto, manual)`) to compare prices.

- What do the p-value and medians suggest?

d) Write one sentence back to the sales rep.

In [ ]:
# a) compare median MSRP by transmission type

# b) compare year by transmission type

# c) compare 2015-2017 medians overall and within each vehicle_size
# For 2015-17 compacts, test automatic vs manual prices with:
# stats.mannwhitneyu(auto, manual)

**Answer:**

### Q7. Significant ≠ important

Among cars whose MPG you trust, 4-door cars get better highway MPG than 2-door cars.

a) Run a Welch t-test. Report the difference in means and the p-value.

b) Translate the difference into dollars. Assume 12,000 miles/year and \$3.50/gallon. How much does the average 4-door driver save per year?

c) Now pretend you only had **30 cars per group.** Draw 30 random 4-door and 30 random 2-door cars, run the same test, and repeat 1,000 times. What fraction of those tests come out significant (p < 0.05)?

d) Would you put "4-door cars are significantly more fuel-efficient" in an ad? What would you say instead?

In [ ]:
# a) Welch t-test, 4-door vs 2-door

# b) yearly gas cost = miles * price_per_gallon / mpg

# c) simulation
n_sims, hits = 1000, 0
for _ in range(n_sims):
    # sample 30 from each group with rng.choice(..., 30, replace=False), run the test, count p < 0.05
    pass

**Answer:**

---
## Part 5: Communicate

### Q8. Your memo to the pricing manager

Write 4–6 sentences. Include:
- Your answer to "do manual cars cost less?", with at least one number
- One other finding from this notebook that matters for pricing
- One limitation of this data a manager should know about

No unexplained jargon: if you say "significant," say what it means.

**Memo:**

---
## 🚀 Stretch (optional)

### 🚀 S1. Horsepower and price

a) Compute the correlation between `engine_hp` and `msrp` three ways: Pearson, Pearson with `np.log10(msrp)`, and Spearman. Why do they differ?

b) Scatter plot HP vs MSRP with a log y-axis (`plt.yscale("log")`).

c) Does adding horsepower *cause* a higher price? Name one lurking variable.

In [ ]:
# S1

### 🚀 S2. Is transmission tied to vehicle size?

For 2015–2017 cars (automatic and manual only), build a crosstab of `vehicle_size` × `transmission_type` and run a chi-square test (`stats.chi2_contingency`). How does this connect to Q6?

In [ ]:
# S2